# NB02 - Preprocessing, caching and quality control
The OASIS-1 processed volumes are already bias-corrected, atlas-registered and brain-masked, so **no second N4 / skull stripping** is done.
Each volume is resized to 88x104x88 (aspect ratio preserved) and z-scored inside the brain mask. QC metrics are computed on the raw volume;
hard failures are excluded and logged, soft outliers are flagged. The cache is saved on Drive, so this runs once.

In [ ]:
# ── 1. Environment Detection (Google Colab vs Local Machine) ──────────────
import sys, os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── 2. Configure Paths ───────────────────────────────────────────────────
if IN_COLAB:
    # Google Colab paths (on your Google Drive)
    PROJECT  = '/content/drive/MyDrive/oasis_project'      # this unzipped project folder on Drive
    DATA_DIR = '/content/drive/MyDrive/OASIS1'             # folder with the 12 OASIS-1 disc archives (+ clinical csv)
    OUT_DIR  = '/content/drive/MyDrive/OASIS1_outputs'     # results are saved here (survives Colab disconnects)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT}/requirements.txt'], check=False)
else:
    # Local machine paths (runs directly on your laptop)
    PROJECT  = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
    DATA_DIR = os.path.join(PROJECT, 'Dataset')
    OUT_DIR  = os.path.join(PROJECT, 'OASIS1_outputs')

# ── 3. install + load ─────────────────────────────────────────────────────
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
from oasis_ad.config import load_config, Paths
from oasis_ad import pipeline as PL, DISCLAIMER
cfg = load_config(os.path.join(PROJECT, 'configs', 'default.yaml'),
                  {'paths.data_dir': DATA_DIR, 'paths.out_dir': OUT_DIR})
paths = Paths(cfg)
import torch, pandas as pd
print(DISCLAIMER)
print('Environment:', 'Google Colab' if IN_COLAB else 'Local Laptop')
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (on Colab switch to Runtime > Change runtime type > T4 GPU)')

In [ ]:
qc, passed = PL.step2_preprocess(cfg, paths)
qc[['participant_id','brain_frac','cov_in','grad_energy','qc_hard_fail','qc_soft_flag','qc_reasons']].sort_values('qc_hard_fail', ascending=False).head(15)

In [ ]:
from IPython.display import Image, display
display(Image(f'{paths.cache_drive}/spot_check.png'))   # visually confirm the volumes look like brains

In [ ]:
print(qc[['qc_hard_fail','qc_soft_flag']].sum())
qc[qc.qc_soft_flag | qc.qc_hard_fail][['participant_id','qc_reasons']]